# SENTINEL — Llama-3-8B QLoRA SFT + GRPO Fine-tune .

Trains an autonomous web-exploitation agent in two stages on Kaggle's free GPU (T4 16GB or P100 16GB):

1. **Stage 1 — QLoRA SFT** on 415 SENTINEL trajectory pairs. Teaches the schema and reasoning patterns. ~45 min.
2. **Stage 2 — GRPO RL alignment** with V01-V18 verifiable reward. Sharpens output quality and JSON validity. ~3-5 hours (may need a second session).

Outputs:
- LoRA adapters pushed to HF Hub (~150MB each)
- WandB runs for both stages
- Smoke-test inference at the end

Prerequisites — run cells in order. See `TRAIN_INSTRUCTIONS.txt` for full setup.


## Cell 1 · Install dependencies

Unsloth pinned for stability. Internet must be ON (Settings → Internet → On).

In [1]:
%%capture
!pip install -q --upgrade pip
# Unsloth manages its own dep resolution. trl floor pinned for stable GRPO API (>=0.15).
!pip install -q --upgrade "unsloth" "trl>=0.15.0,<0.18" peft accelerate bitsandbytes datasets
!pip install -q wandb huggingface_hub
print("deps installed")

## Cell 2 · Imports + GPU check

In [2]:
import os, json, gc, re, time, math, random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List
import torch
import pandas as pd
from datasets import load_dataset, Dataset
import wandb
from huggingface_hub import login as hf_login

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB)")
else:
    raise RuntimeError("No GPU detected. Settings → Accelerator → GPU T4 x1")

PyTorch: 2.10.0+cu128
GPU: Tesla T4 (15.6 GB)


## Cell 3 · Configuration

**EDIT `cfg.dataset_path` and `cfg.hf_repo_*` to match your setup.**

Toggle `cfg.run_grpo` if you want to skip Stage 2 (or run it separately later).

In [3]:
@dataclass
class Config:
    # ---- Paths (EDIT these) ----
    dataset_path: str = "/kaggle/input/sentinel-dataset"
    train_file: str = "train_llama3.jsonl"
    val_file: str = "train_llama3_val.jsonl"
    output_dir: str = "/kaggle/working"

    # ---- Model ----
    base_model: str = "unsloth/llama-3-8b-Instruct-bnb-4bit"
    max_seq_length: int = 4096

    # ---- LoRA (calibrated for 415 samples; small r prevents overfit) ----
    lora_rank: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # ---- SFT hyperparams ----
    sft_epochs: int = 2
    sft_per_device_batch: int = 2
    sft_grad_accum: int = 4
    sft_lr: float = 2e-4
    sft_warmup_ratio: float = 0.05
    sft_weight_decay: float = 0.01
    sft_neftune_alpha: float = 5.0      # embedding noise; helps small-data generalization

    # ---- GRPO hyperparams ----
    run_grpo: bool = True               # set False to skip Stage 2
    grpo_max_steps: int = 200
    grpo_per_device_batch: int = 1
    grpo_grad_accum: int = 4
    grpo_lr: float = 5e-6               # 40x smaller than SFT lr (RL is delicate)
    grpo_num_generations: int = 4       # G = group size; smaller fits Kaggle T4
    grpo_max_prompt_length: int = 1024
    grpo_max_completion_length: int = 512
    grpo_kl_beta: float = 0.04          # KL penalty keeps model close to SFT init
    grpo_temperature: float = 0.9

    # ---- WandB / HF (EDIT repo names) ----
    wandb_project: str = "sentinel-llama3-8b"
    hf_repo_sft: str = "niranjan2777/sentinel-llama3-8b-sft"
    hf_repo_grpo: str = "niranjan2777/sentinel-llama3-8b-grpo"
    hf_private: bool = True             # private by default; flip to False to publish

    # ---- Reproducibility ----
    seed: int = 42

cfg = Config()
random.seed(cfg.seed); torch.manual_seed(cfg.seed)

assert not cfg.hf_repo_sft.startswith("YOUR-HF-USERNAME"), \
    "EDIT cfg.hf_repo_sft and cfg.hf_repo_grpo to your actual HF username"
print("config loaded")
for k, v in vars(cfg).items(): print(f"  {k:30s} = {v}")

config loaded
  dataset_path                   = /kaggle/input/sentinel-dataset
  train_file                     = train_llama3.jsonl
  val_file                       = train_llama3_val.jsonl
  output_dir                     = /kaggle/working
  base_model                     = unsloth/llama-3-8b-Instruct-bnb-4bit
  max_seq_length                 = 4096
  lora_rank                      = 16
  lora_alpha                     = 32
  lora_dropout                   = 0.0
  target_modules                 = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
  sft_epochs                     = 2
  sft_per_device_batch           = 2
  sft_grad_accum                 = 4
  sft_lr                         = 0.0002
  sft_warmup_ratio               = 0.05
  sft_weight_decay               = 0.01
  sft_neftune_alpha              = 5.0
  run_grpo                       = True
  grpo_max_steps                 = 200
  grpo_per_device_batch          = 1
  grpo_grad_accum            

## Cell 4 · Authenticate WandB + HuggingFace

Reads tokens from Kaggle Secrets:
- `HF_TOKEN` — HuggingFace write token
- `WANDB_API_KEY` — WandB API key

Add both via Add-ons → Secrets in the Kaggle sidebar.

In [4]:
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    wandb_token = secrets.get_secret("WANDB_API_KEY")
except Exception as e:
    raise RuntimeError(
        "Could not read Kaggle Secrets HF_TOKEN and WANDB_API_KEY.\n"
        "Setup: Add-ons -> Secrets -> Add a new secret\n"
        "  Label: HF_TOKEN          Value: hf_xxx... (HuggingFace WRITE token)\n"
        "  Label: WANDB_API_KEY     Value: your wandb API key from wandb.ai/authorize\n"
        f"Underlying error: {e}"
    )

hf_login(token=hf_token, add_to_git_credential=True)
wandb.login(key=wandb_token)
print("authenticated to HF + WandB")

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pathfixed (pathfixed-rav) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


authenticated to HF + WandB


## Cell 5 · Load + verify dataset

Reads `train_llama3.jsonl` and `train_llama3_val.jsonl` from the Kaggle dataset path. Asserts the chat-messages structure is intact.

In [5]:
train_path = Path(cfg.dataset_path) / cfg.train_file
val_path = Path(cfg.dataset_path) / cfg.val_file

  # Auto-discover if the configured path doesn't exist (handles Kaggle's variable mount layouts)
if not train_path.exists() or not val_path.exists():
    print(f"configured path {cfg.dataset_path} not found — searching /kaggle/input ...")
    found = {}
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f == cfg.train_file:
                found["train"] = Path(root) / f
            elif f == cfg.val_file:
                found["val"] = Path(root) / f
    assert "train" in found and "val" in found, \
        f"could not locate {cfg.train_file} / {cfg.val_file} anywhere under /kaggle/input"
    train_path = found["train"]
    val_path = found["val"]
    cfg.dataset_path = str(train_path.parent)   # update for downstream cells
    print(f"  found train: {train_path}")
    print(f"  found val:   {val_path}")
    print(f"  dataset_path updated to: {cfg.dataset_path}")

assert train_path.exists(), f"missing: {train_path}"
assert val_path.exists(), f"missing: {val_path}"
train_ds = load_dataset("json", data_files=str(train_path), split="train")
val_ds = load_dataset("json", data_files=str(val_path), split="train")

  # Schema sanity
for split_name, ds in [("train", train_ds), ("val", val_ds)]:
    s0 = ds[0]
    assert "messages" in s0, f"{split_name}: 'messages' field missing"
    roles = [m["role"] for m in s0["messages"]]
    assert roles == ["system", "user", "assistant"], f"{split_name}: unexpected roles {roles}"

print(f"\ntrain: {len(train_ds)} samples")
print(f"val:   {len(val_ds)} samples")
print(f"sample 0 user msg (200 chars): {train_ds[0]['messages'][1]['content'][:200]}...")
print(f"sample 0 assistant msg (200 chars): {train_ds[0]['messages'][2]['content'][:200]}...")

configured path /kaggle/input/sentinel-dataset not found — searching /kaggle/input ...
  found train: /kaggle/input/datasets/niranjankagglecom/sentinel-dataset/train_llama3.jsonl
  found val:   /kaggle/input/datasets/niranjankagglecom/sentinel-dataset/train_llama3_val.jsonl
  dataset_path updated to: /kaggle/input/datasets/niranjankagglecom/sentinel-dataset


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]


train: 415 samples
val:   21 samples
sample 0 user msg (200 chars): GOAL: Diagnose Cloudflare 1015 rate limit

HTML_SNIPPET: (empty — see prior_turns for context)

PRIOR_TURNS (1):

--- Turn 1 ---
Thought: PHP search probe.
Action: SQL_INJECT
Action_Input: {"target_ur...
sample 0 assistant msg (200 chars): {"Thought":"Turn 1 Cloudflare 1015 rate-limit (Retry-After 30s). PHP backend behind Cloudflare. Cannot continue from this IP. Wait 30s minimum or rotate to different source IP for further probing.","A...


In [6]:
import os
print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        print(f"  {p}  ({size/1024:.1f} KB)")

=== /kaggle/input contents ===
  /kaggle/input/datasets/niranjankagglecom/sentinel-dataset/train_llama3.jsonl  (999.7 KB)
  /kaggle/input/datasets/niranjankagglecom/sentinel-dataset/seed_samples.jsonl  (553.6 KB)
  /kaggle/input/datasets/niranjankagglecom/sentinel-dataset/train_llama3_val.jsonl  (51.0 KB)


## Cell 6 · V01-V18 scorer (used for eval metrics + GRPO reward)

Continuous 0.0–1.0 score derived from the validator. Used in two places:
1. As a metric during eval to track output quality alongside loss
2. As the reward function in the GRPO stage

In [7]:
ACTIONS = {"SQL_INJECT", "XSS_INJECT", "RETRY_MUTATED", "ANALYZE_RESPONSE", "CRAWL_DEEPER", "WAIT", "STOP"}
PAYLOAD_ACTIONS = {"SQL_INJECT", "XSS_INJECT", "RETRY_MUTATED"}
BACKENDS_TOK = ["php", "mysql", "aspx", "mssql", "asp.net", "postgres", "jsp", "java", "oracle",
                "node", "angular", "spa", "sqlite", "mongo", "dvwa", "juice", "wordpress"]
CTX_TOK = ["attribute", "script", "html", "url", "json", "template", "sql-numeric", "sql-string",
           "numeric context", "string context", "innerhtml"]
FORBIDDEN = ["this might", "this could", "appears to", "may be vulnerable",
             "let me", "i think", "possibly", "seems to"]
VALID_MUT = {"encoding_bypass", "comment_injection", "hex_substitution", "polyglot_swap",
             "case_mutation", "char_concatenation", "double_encoding", "unicode_escape", "column_count_fix"}
VALID_SIG = {"fail_filtered", "fail_blocked", "fail_no_reflection", "fail_column_mismatch",
             "partial_success", "success_authenticated", "success_data_leaked", "ambiguous"}
VALID_SUC = {"authenticated_dashboard", "jwt_issued", "admin_panel_accessed",
             "data_exfiltrated", "session_established", "privilege_escalated"}


def score_output(text: str) -> float:
    """V01-V18-style continuous scorer. Returns a float in [0.0, 1.0]."""
    if not text or not isinstance(text, str):
        return 0.0
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```\s*$", "", cleaned)
    try:
        obj = json.loads(cleaned)
    except Exception:
        return 0.0
    if not isinstance(obj, dict):
        return 0.05
    
    score = 0.10  # parses as JSON
    if {"Thought", "Action", "Action_Input"} == set(obj.keys()):
        score += 0.15
    else:
        return score
    
    action = obj.get("Action", "")
    if action in ACTIONS:
        score += 0.10
    
    ai = obj.get("Action_Input", {})
    if isinstance(ai, dict) and {"target_url", "method", "parameters", "headers", "rationale"}.issubset(ai.keys()):
        score += 0.15
    
    thought = obj.get("Thought", "")
    tlow = thought.lower()
    n_words = len(thought.split())
    if 8 <= n_words <= 90:
        score += 0.10
    if not any(f in tlow for f in FORBIDDEN):
        score += 0.05
    
    if action in PAYLOAD_ACTIONS:
        if any(b in tlow for b in BACKENDS_TOK):
            score += 0.10
        if action != "RETRY_MUTATED" and any(c in tlow for c in CTX_TOK):
            score += 0.05
        if isinstance(ai, dict) and isinstance(ai.get("parameters"), dict) and len(ai["parameters"]) > 0:
            score += 0.10
        if action == "RETRY_MUTATED" and isinstance(ai, dict) and ai.get("mutation_class") in VALID_MUT:
            score += 0.10
    elif action == "STOP":
        if isinstance(ai, dict):
            if ai.get("success_state") in VALID_SUC:
                score += 0.10
            if ai.get("evidence") and len(ai.get("evidence", "")) >= 20:
                score += 0.10
    elif action == "ANALYZE_RESPONSE":
        if isinstance(ai, dict):
            if ai.get("signal") in VALID_SIG:
                score += 0.10
            if ai.get("next_recommended") in ACTIONS:
                score += 0.10
    elif action in {"WAIT", "CRAWL_DEEPER"}:
        if n_words >= 15:
            score += 0.10
    
    return min(score, 1.0)


# Sanity: training assistant outputs should score high
test_scores = [score_output(s["messages"][2]["content"]) for s in [train_ds[i] for i in range(5)]]
print(f"first 5 train assistant scores: {test_scores}")
assert all(s > 0.7 for s in test_scores), "training data should score >0.7 — check assistant content"

first 5 train assistant scores: [0.85, 0.9, 0.9, 0.95, 0.9]


## Cell 7 · Load Llama-3-8B (4-bit)

Unsloth's 4-bit Llama-3-8B-Instruct. ~4 GB on GPU after load.

In [8]:
pip install unsloth_zoo

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip uninstall -y unsloth unsloth_zoo
!pip install --no-deps --upgrade "unsloth_zoo"
!pip install --no-deps --upgrade "unsloth"
!pip install --no-deps "trl>=0.15.0,<0.18" "peft>=0.13" "accelerate>=1.0" bitsandbytes
print("clean reinstall done — RESTART KERNEL NOW")

Found existing installation: unsloth 2025.10.8
Uninstalling unsloth-2025.10.8:
  Successfully uninstalled unsloth-2025.10.8
Found existing installation: unsloth_zoo 2025.10.9
Uninstalling unsloth_zoo-2025.10.9:
  Successfully uninstalled unsloth_zoo-2025.10.9
  Using cached unsloth_zoo-2026.5.1-py3-none-any.whl.metadata (32 kB)
  Using cached unsloth-2026.5.2-py3-none-any.whl.metadata (56 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 43.3 MB/s  0:00:016m0:00:0100:01
clean reinstall done — RESTART KERNEL NOW


In [10]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# Llama-3 tokenizer has no default pad_token; set explicitly to silence collator warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"loaded {cfg.base_model}")
print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-12 02:49:41.103351: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778554181.501149      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778554181.604672      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778554182.603618      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778554182.603667      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778554182.603670      57 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

loaded unsloth/llama-3-8b-Instruct-bnb-4bit
pad_token: '<|reserved_special_token_250|>' (id=128255)
VRAM: 5.71 GB / 15.64 GB


## Cell 8 · Apply LoRA adapters

r=16, alpha=32 on all attention + MLP linear layers. Trainable params should be ~0.5% of total.

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.target_modules),
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg.seed,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## Cell 9 · Apply chat template to dataset

Materializes the Llama-3 chat template into a `text` field. SFTTrainer reads `text`, computes loss only on assistant tokens (loss masking handled by unsloth).

In [12]:
def apply_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples["messages"]
        ]
    }

train_ds = train_ds.map(apply_template, batched=True)
val_ds = val_ds.map(apply_template, batched=True)

print(f"templated train: {len(train_ds)}")
print("=== sample (first 400 chars) ===")
print(train_ds[0]["text"][:400])

lengths = [len(tokenizer.encode(t)) for t in train_ds["text"][:50]]
print(f"\ntoken length p50/p95/max (first 50): {sorted(lengths)[25]} / {sorted(lengths)[47]} / {max(lengths)}")

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

templated train: 415
=== sample (first 400 chars) ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are SENTINEL, an autonomous web-exploitation agent. Given an HTML snippet and a goal (and optionally prior agent turns), you reason about vulnerabilities and emit a single JSON action that advances the exploit loop:

observe -> identify attack surface -> select exploit -> generate payload -> interpret response -> adapt and retry -> d

token length p50/p95/max (first 50): 532 / 637 / 782


## Cell 10 · Generation callback (eval-time visibility)

Each eval step, generate 2 sample completions and log to WandB as a table — see what the model is actually producing, not just the loss number.

In [13]:
from transformers import TrainerCallback

class GenerationCallback(TrainerCallback):
    """Each eval step: generate 2 sample completions, score them, log to WandB."""
    def __init__(self, tokenizer, val_ds):
        self.tokenizer = tokenizer
        self.val_ds = val_ds

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        try:
            FastLanguageModel.for_inference(model)
            rows = []
            sample_ids = [0, len(self.val_ds) // 2]
            for i in sample_ids:
                msgs = self.val_ds[i]["messages"][:-1]
                prompt = self.tokenizer.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True
                )
                inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=400,
                        do_sample=False,                       # greedy = deterministic
                        pad_token_id=self.tokenizer.eos_token_id,
                    )
                gen = self.tokenizer.decode(
                    out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
                )
                ref = self.val_ds[i]["messages"][-1]["content"]
                rows.append({
                    "step": state.global_step,
                    "sample_idx": i,
                    "prompt_tail": prompt[-300:],
                    "generated": gen,
                    "reference": ref[:400],
                    "score": score_output(gen),
                })
            if wandb.run is not None:
                wandb.log(
                    {
                        "eval/sample_table": wandb.Table(dataframe=pd.DataFrame(rows)),
                        "eval/mean_sample_score": sum(r["score"] for r in rows) / len(rows),
                    },
                    step=state.global_step,
                )
            FastLanguageModel.for_training(model)
        except Exception as e:
            print(f"[GenerationCallback] error: {e}")
            try:
                FastLanguageModel.for_training(model)
            except Exception:
                pass

print("callback ready")

callback ready


## Cell 11 · SFT trainer setup

- BF16 (Llama-3 native, FP16 causes loss spikes)
- Cosine LR + 5% warmup
- Paged AdamW 8-bit (memory efficient)
- Gradient checkpointing (essential for Kaggle 16GB)
- Save best by eval_loss
- NEFTune α=5 (small-data regularizer)
- packing=False (preserves sample boundaries for structured outputs)

In [14]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Auto-detect precision: T4/V100/P100 → FP16; A100/H100 → BF16
_use_bf16 = torch.cuda.is_bf16_supported()
_use_fp16 = not _use_bf16

print(
    f"precision: bf16={_use_bf16}, "
    f"fp16={_use_fp16} on {torch.cuda.get_device_name(0)}"
)

wandb.init(
    project=cfg.wandb_project,
    name=f"sft-r{cfg.lora_rank}-a{cfg.lora_alpha}-lr{cfg.sft_lr}",
    config={k: str(v) for k, v in vars(cfg).items()},
    reinit=True,
)

sft_args = SFTConfig(
    output_dir=f"{cfg.output_dir}/sft",
    num_train_epochs=cfg.sft_epochs,

    per_device_train_batch_size=cfg.sft_per_device_batch,
    per_device_eval_batch_size=cfg.sft_per_device_batch,
    gradient_accumulation_steps=cfg.sft_grad_accum,

    learning_rate=cfg.sft_lr,
    warmup_ratio=cfg.sft_warmup_ratio,
    lr_scheduler_type="cosine",
    weight_decay=cfg.sft_weight_decay,

    optim="paged_adamw_8bit",

    bf16=_use_bf16,
    fp16=_use_fp16,

    gradient_checkpointing=True,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=20,

    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="wandb",
    run_name=f"sft-r{cfg.lora_rank}",

    seed=cfg.seed,

    neftune_noise_alpha=cfg.sft_neftune_alpha,

    max_seq_length=cfg.max_seq_length,
    dataset_text_field="text",

    packing=False,
    dataloader_num_workers=2,

    # FP16 stability
    max_grad_norm=1.0,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_args,
)

# CRITICAL: mask loss on system + user tokens only
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

sft_trainer.add_callback(
    GenerationCallback(tokenizer, val_ds)
)

print("SFT trainer ready (loss masked to assistant tokens only)")

precision: bf16=False, fp16=True on Tesla T4


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
/kaggle/working/unsloth_compiled_cache/UnslothSFTTrainer.py:821: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/415 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/21 [00:00<?, ? examples/s]

...

Map (num_proc=8):   0%|          | 0/415 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/415 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/21 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/21 [00:00<?, ? examples/s]

SFT trainer ready (loss masked to assistant tokens only)


## Cell 12 · Run SFT training

Expected: ~30-60 min on T4, ~20-40 min on P100. Monitor in WandB while it runs.

In [15]:
t0 = time.time()
sft_stats = sft_trainer.train()
print(f"\nSFT done in {(time.time()-t0)/60:.1f} min")
print(f"final train_loss: {sft_stats.training_loss:.4f}")
print(f"VRAM peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 415 | Num Epochs = 2 | Total steps = 104
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,1.333500,1.492488
40,1.127200,1.263251
60,0.689200,1.210084
80,0.724800,1.163273
100,0.763000,1.146508


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



SFT done in 28.1 min
final train_loss: 1.0764
VRAM peak: 7.85 GB


In [16]:
!nvidia-sm

/bin/bash: line 1: nvidia-sm: command not found


In [17]:
print("alive")

alive


In [24]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    if "adapter_model.safetensors" in files:
        print("FOUND ADAPTER:", root)

    for d in dirs:
        if "checkpoint" in d.lower():
            print("FOUND CHECKPOINT:", os.path.join(root, d))

FOUND ADAPTER: /kaggle/working/sft_adapter
FOUND CHECKPOINT: /kaggle/working/sft/checkpoint-104
FOUND CHECKPOINT: /kaggle/working/sft/checkpoint-100
FOUND ADAPTER: /kaggle/working/sft/checkpoint-104
FOUND ADAPTER: /kaggle/working/sft/checkpoint-100


## Cell 13 · Save SFT adapter and push to HF Hub

Adapter-only push (~150MB). Model card auto-generated by unsloth/peft.

In [27]:
import shutil

# Zip the trained adapter folder
shutil.make_archive(
    "/kaggle/working/sft_adapter",
    "zip",
    "/kaggle/working/sft_adapter"
)

print("ZIP created at:")
print("/kaggle/working/sft_adapter.zip")

ZIP created at:
/kaggle/working/sft_adapter.zip


In [29]:
import shutil
import os

source_dir = "/kaggle/working/sft_adapter"
zip_path = "/kaggle/working/sft_adapter.zip"

# Create zip
shutil.make_archive(
    "/kaggle/working/sft_adapter",
    "zip",
    source_dir
)

print("ZIP exists:", os.path.exists(zip_path))
print("Saved at:", zip_path)


ZIP exists: True
Saved at: /kaggle/working/sft_adapter.zip


In [31]:
from IPython.display import Javascript

display(Javascript('window.open("/files/sft_adapter.zip")'))

<IPython.core.display.Javascript object>

In [26]:
sft_local = f"{cfg.output_dir}/sft_adapter"
sft_trainer.save_model(sft_local)
tokenizer.save_pretrained(sft_local)
print(f"SFT adapter saved locally: {sft_local}")

try:
    model.push_to_hub(cfg.hf_repo_sft, token=hf_token, private=cfg.hf_private)
    tokenizer.push_to_hub(cfg.hf_repo_sft, token=hf_token, private=cfg.hf_private)
    visibility = "private" if cfg.hf_private else "public"
    print(f"pushed ({visibility}) to https://huggingface.co/{cfg.hf_repo_sft}")
except Exception as e:
    print(f"HF push failed: {e}")
    print("Adapter saved locally; you can push manually later via huggingface_hub.HfApi().")

wandb.finish()

SFT adapter saved locally: /kaggle/working/sft_adapter
HF push failed: (Request ID: Root=1-6a02a14a-0cb184703b8af05200b46bf0;fc83b134-616d-49da-9db9-c42213e977c0)

403 Forbidden: You don't have the rights to create a model under the namespace "niranjan2777".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.
Adapter saved locally; you can push manually later via huggingface_hub.HfApi().


In [35]:

from huggingface_hub import login

login(token=hf_token)
repo_id = "niranjan2777/sentinel_sft"

model.push_to_hub(
    repo_id=repo_id,
    token=hf_token,
)

tokenizer.push_to_hub(
    repo_id=repo_id,
    token=hf_token,
)

print("Upload successful!")

HfHubHTTPError: (Request ID: Root=1-6a02a48c-0974d7520743880a19d8d700;74d157e2-24a5-44af-9728-1254ff6f88ed)

403 Forbidden: Authorization error..
Cannot access content at: https://huggingface.co/niranjan2777/sentinel_sft.git/info/lfs/objects/batch.
Make sure your token has the correct permissions.

## Cell 14 · SFT smoke-test inference

Run a hand-crafted prompt through the SFT model and score it. This is the baseline for whether GRPO is needed.

In [20]:
test_msgs = [
    {"role": "system", "content": train_ds[0]["messages"][0]["content"]},  # reuse system prompt
    {"role": "user", "content": (
        "GOAL: Bypass authentication and reach admin dashboard\n\n"
        "HTML_SNIPPET:\n"
        "<form method=\"POST\" action=\"/login.php\" id=\"loginForm\">\n"
        "  <input type=\"text\" name=\"username\" autocomplete=\"off\">\n"
        "  <input type=\"password\" name=\"password\">\n"
        "  <input type=\"hidden\" name=\"redirect\" value=\"/admin/dashboard\">\n"
        "  <button type=\"submit\">Sign In</button>\n"
        "</form>"
    )},
]
FastLanguageModel.for_inference(model)
prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== SFT MODEL OUTPUT ===")
print(gen)
print(f"\nscore: {score_output(gen):.3f}  (1.0 is perfect)")

=== SFT MODEL OUTPUT ===
{"Thought":"POST /login.php with username text input — PHP/MySQL backend. SQL-string context. OR-tautology in username with comment terminator; redirect preserved.","Action":"SQL_INJECT","Action_Input":{"target_url":"/login.php","method":"POST","parameters":{"username":"admin' OR '1'='1' -- ","password":"x","redirect":"/admin/dashboard"},"headers":{"Content-Type":"application/x-www-form-urlencoded"},"rationale":"MySQL tautology bypasses password check; redirect preserved"}}

score: 0.900  (1.0 is perfect)


In [36]:
!pip -q install PyDrive


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [37]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()

drive = GoogleDrive(gauth)

print("Google Drive authenticated")

Google Drive authenticated


In [41]:
import os

print(os.listdir("/kaggle/working"))

['huggingface_tokenizers_cache', 'unsloth_compiled_cache', 'sft_adapter', 'wandb', 'grpo', 'sft', '.virtual_documents', 'sft_adapter.zip']


In [38]:
file_path = "/kaggle/working/sft_adapter.zip"

gfile = drive.CreateFile({
    "title": "sft_adapter.zip"
})

gfile.SetContentFile(file_path)
gfile.Upload()

print("Uploaded successfully!")
print("Drive File ID:", gfile["id"])

RedirectMissingLocation: Redirected but the response is missing a Location: header.

In [ ]:
print(
    f"https://drive.google.com/file/d/{gfile['id']}/view"
)

## ── Stage 2 — GRPO RL alignment ──

Optional. Set `cfg.run_grpo=False` in Cell 3 to stop after SFT.

GRPO trains the model to *optimize* for the V01-V18 reward instead of just imitating training data. Generates G=4 outputs per prompt, scores each, updates the policy toward higher-scoring outputs.

Time on Kaggle T4: ~3-5 hours. May exceed a single session — checkpoints save every 50 steps.

## Cell 15 · GRPO data prep — extract prompts only

In [21]:
if cfg.run_grpo:
    def to_prompt(example):
        msgs = example["messages"][:-1]   # drop assistant
        return {"prompt": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)}
    
    grpo_train = train_ds.map(to_prompt, remove_columns=train_ds.column_names)
    print(f"GRPO train prompts: {len(grpo_train)}")
    print(f"sample prompt (first 300 chars): {grpo_train[0]['prompt'][:300]}")
else:
    print("GRPO disabled (cfg.run_grpo=False)")

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

GRPO train prompts: 415
sample prompt (first 300 chars): <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are SENTINEL, an autonomous web-exploitation agent. Given an HTML snippet and a goal (and optionally prior agent turns), you reason about vulnerabilities and emit a single JSON action that advances the exploit loop:

observe -> identif


## Cell 16 · GRPO reward function

Wraps `score_output` for GRPO's expected signature.

In [22]:
if cfg.run_grpo:
    def grpo_reward_fn(prompts, completions, **kwargs):
        """GRPO calls this with a batch of prompts and completions; returns a list of rewards."""
        scores = [score_output(c) for c in completions]
        return scores
    
    # Sanity check the reward function
    sample_completion = train_ds[0]["messages"][2]["content"]
    print(f"reward on a perfect training sample: {grpo_reward_fn(['x'], [sample_completion])[0]:.3f}")
    print(f"reward on garbage: {grpo_reward_fn(['x'], ['this is not json'])[0]:.3f}")

reward on a perfect training sample: 0.850
reward on garbage: 0.000


## Cell 17 · GRPO trainer setup + run

Picks up from the SFT-trained model already in memory. β=0.04 KL keeps it close to SFT init.

If you split into a 2nd Kaggle session, comment out this cell's training and instead reload the SFT adapter from HF (see TRAIN_INSTRUCTIONS.txt).

In [23]:
if cfg.run_grpo:
    from trl import GRPOTrainer, GRPOConfig

    FastLanguageModel.for_training(model)
    wandb.init(
        project=cfg.wandb_project,
        name=f"grpo-r{cfg.lora_rank}-lr{cfg.grpo_lr}-G{cfg.grpo_num_generations}",
        config={k: str(v) for k, v in vars(cfg).items()},
        reinit=True,
    )

    grpo_args = GRPOConfig(
        output_dir=f"{cfg.output_dir}/grpo",
        max_steps=cfg.grpo_max_steps,
        per_device_train_batch_size=cfg.grpo_per_device_batch,
        gradient_accumulation_steps=cfg.grpo_grad_accum,
        learning_rate=cfg.grpo_lr,
        lr_scheduler_type="constant_with_warmup",
        warmup_ratio=0.05,
        optim="paged_adamw_8bit",
        bf16=_use_bf16,
        fp16=_use_fp16,
        gradient_checkpointing=True,
        num_generations=cfg.grpo_num_generations,
        max_prompt_length=cfg.grpo_max_prompt_length,
        max_completion_length=cfg.grpo_max_completion_length,
        beta=cfg.grpo_kl_beta,
        temperature=cfg.grpo_temperature,
        logging_steps=2,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        report_to="wandb",
        run_name=f"grpo-r{cfg.lora_rank}",
        seed=cfg.seed,
    )

    # TRL>=0.16 renamed `tokenizer` to `processing_class`. Try the new API first,
    # fall back to the old one for compatibility with whatever unsloth installed.
    try:
        grpo_trainer = GRPOTrainer(
            model=model,
            processing_class=tokenizer,
            reward_funcs=[grpo_reward_fn],
            train_dataset=grpo_train,
            args=grpo_args,
        )
    except TypeError:
        grpo_trainer = GRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            reward_funcs=[grpo_reward_fn],
            train_dataset=grpo_train,
            args=grpo_args,
        )

    t0 = time.time()
    grpo_stats = grpo_trainer.train()
    print(f"\nGRPO done in {(time.time()-t0)/60:.1f} min")
else:
    print("skipped")

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 415 | Num Epochs = 2 | Total steps = 200
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


AttributeError: 'UnslothGRPOTrainer' object has no attribute 'image_token_id'

## Cell 18 · Save GRPO adapter and push to HF Hub

In [ ]:
if cfg.run_grpo:
    grpo_local = f"{cfg.output_dir}/grpo_adapter"
    grpo_trainer.save_model(grpo_local)
    tokenizer.save_pretrained(grpo_local)
    print(f"GRPO adapter saved: {grpo_local}")
    try:
        model.push_to_hub(cfg.hf_repo_grpo, token=hf_token, private=cfg.hf_private)
        tokenizer.push_to_hub(cfg.hf_repo_grpo, token=hf_token, private=cfg.hf_private)
        visibility = "private" if cfg.hf_private else "public"
        print(f"pushed ({visibility}) to https://huggingface.co/{cfg.hf_repo_grpo}")
    except Exception as e:
        print(f"HF push failed: {e}")
    wandb.finish()

## Cell 19 · Final smoke-test inference

Same prompt as SFT smoke test — compare output and score against the SFT-only result.

In [ ]:
FastLanguageModel.for_inference(model)
prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id)
gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== FINAL MODEL OUTPUT ===")
print(gen)
print(f"\nscore: {score_output(gen):.3f}  (1.0 is perfect)")

print("\n\n=== TRAINING COMPLETE ===")
print(f"SFT  adapter: https://huggingface.co/{cfg.hf_repo_sft}")
if cfg.run_grpo:
    print(f"GRPO adapter: https://huggingface.co/{cfg.hf_repo_grpo}")
print("For local inference on RTX 3050 6GB, see TRAIN_INSTRUCTIONS.txt — Section 8")